In [1]:
import pandas as pd

df = pd.read_csv("final_health_dataset.csv")

print(df.head())

   age  headache  nausea  glucose         bmi  stress  sleep  label
0   25         1       1      104   24.592649       3      4      0
1   38         0       0       86  430.581811       5      7      0
2   72         0       0       71  367.346939       5      7      2
3   22         1       1      104   25.563870       0      5      0
4   58         0       0       85  430.612245       5      7      0


In [2]:
df.head()

,age,headache,nausea,glucose,bmi,stress,sleep,label
0,25,1,1,104,24.592649,3,4,0
1,38,0,0,86,430.581811,5,7,0
2,72,0,0,71,367.346939,5,7,2
3,22,1,1,104,25.563870,0,5,0
4,58,0,0,85,430.612245,5,7,0


In [3]:
df.tail()

,age,headache,nausea,glucose,bmi,stress,sleep,label
998,29,1,1,94,21.646578,3,7,0
999,56,0,0,110,275.748722,5,7,2
1000,50,0,0,255,634.794684,5,7,2
1001,62,0,0,206,463.905325,5,7,2
1002,54,0,0,96,414.201183,5,7,0


In [4]:
print(df.shape)
print(df.columns)
df.info()

(1003, 8)
Index(['age', 'headache', 'nausea', 'glucose', 'bmi', 'stress', 'sleep',
       'label'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1003 entries, 0 to 1002
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1003 non-null   int64  
 1   headache  1003 non-null   int64  
 2   nausea    1003 non-null   int64  
 3   glucose   1003 non-null   int64  
 4   bmi       997 non-null    float64
 5   stress    1003 non-null   int64  
 6   sleep     1003 non-null   int64  
 7   label     1003 non-null   int64  
dtypes: float64(1), int64(7)
memory usage: 62.8 KB


In [5]:
from sklearn.model_selection import train_test_split

X = df.drop("label", axis=1)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
print(df.isnull().sum())

age         0
headache    0
nausea      0
glucose     0
bmi         6
stress      0
sleep       0
label       0
dtype: int64


In [8]:
df["bmi"].fillna(df["bmi"].mean(), inplace=True)

In [9]:
print(df.isnull().sum())

age         0
headache    0
nausea      0
glucose     0
bmi         0
stress      0
sleep       0
label       0
dtype: int64


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = df.drop("label", axis=1)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9552238805970149


In [11]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98       186
           2       0.75      0.60      0.67        15

    accuracy                           0.96       201
   macro avg       0.86      0.79      0.82       201
weighted avg       0.95      0.96      0.95       201



In [12]:
import joblib

joblib.dump(model, "model.pkl")

['model.pkl']

In [ ]:
from flask import Flask, render_template, request
import joblib
import numpy as np

app = Flask(__name__)

# Load model
model = joblib.load("model.pkl")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    try:
        age = float(request.form["age"])
        headache = int(request.form["headache"])
        nausea = int(request.form["nausea"])
        glucose = float(request.form["glucose"])
        bmi = float(request.form["bmi"])
        stress = float(request.form["stress"])
        sleep = float(request.form["sleep"])

        data = np.array([[age, headache, nausea, glucose, bmi, stress, sleep]])
        prediction = model.predict(data)[0]

        if prediction == 0:
            result = "Normal"
        elif prediction == 1:
            result = "Migraine"
        else:
            result = "Diabetes"

        return render_template("result.html", result=result)

    except Exception as e:
        return str(e)

if __name__ == "__main__":
    app.run(debug=True, use_reloader=False, port=5001)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [05/Apr/2026 18:33:27] "GET / HTTP/1.1" 500 -
Traceback (most recent call last):
  File "c:\Users\Lenovo\anaconda3\Lib\site-packages\flask\app.py", line 2552, in __call__
    return self.wsgi_app(environ, start_response)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Lenovo\anaconda3\Lib\site-packages\flask\app.py", line 2532, in wsgi_app
    response = self.handle_exception(e)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Lenovo\anaconda3\Lib\site-packages\flask\app.py", line 2529, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Lenovo\anaconda3\Lib\site-packages\flask\app.py", line 1825, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Lenovo\anaconda3\Lib\site-packages\flask\app.py", line 1823, in full_dispatch_request
    rv = self.dispatch_req